### `capture_to_pose_valeurs_dans_csv.ipynb`

- **Input**: A folder containing `.jpg`, `.png`, or `.jpeg` images of full-body screenshots from stand-up shows.  
- **Output**: One CSV file per image, saved in the output folder, named `image_name-poses_detectees.csv`. Each row contains:  
  - Image name  
  - Bounding box coordinates (`bbox_xmin`, `bbox_ymin`, `bbox_xmax`, `bbox_ymax`)  
  - 2D coordinates (`x`, `y`) for each keypoint, following this order:  
    `Nez`, `Oeil_2`, `Oeil_1`, `Oreille_1`, `Oreille_2`, `Epaule_2`, `Epaule_1`, `Coude_2`, `Coude_1`, `Poignet_2`, `Poignet_1`, `Hanche_2`, `Hanche_1`, `Genou_2`, `Genou_1`, `Cheville_2`, `Cheville_1`

- **Purpose**: Perform pose estimation on a batch of images using `yolov8l-pose.pt` and extract raw, **non-normalized** keypoint coordinates and bounding boxes into separate CSVs for each image.


Below is a script that:

Takes as input a single image file

Performs pose detection using a pretrained model

Outputs a .csv file named after the input image, with -poses_detected appended

Each row corresponds to one image and includes:

The image name

One column per corner of the bounding box (with its coordinates)

One column per keypoint (with its x, y coordinates)

Keypoint information:
The keypoints correspond to body parts in the following order:

Eye_2

Eye_1

Ear_1

Ear_2

Shoulder_2

Shoulder_1

Elbow_2

Elbow_1

Wrist_2

Wrist_1

Hip_2

Hip_1

Knee_2

Knee_1

Ankle_2

Ankle_1

The coordinates are not normalized.

In [ ]:
import os
import csv
from ultralytics import YOLO

# Load the pose detection model
model = YOLO("yolov8l-pose.pt")

# Set input/output folders
image_folder =  # TODO: set the path to the input folder
output_folder =  # TODO: set the path to the output folder

# List of keypoint names (in French, matching output files)
keypoints_names = [
    "Nez", "Oeil_2", "Oeil_1", "Oreille_1", "Oreille_2",
    "Epaule_2", "Epaule_1", "Coude_2", "Coude_1",
    "Poignet_2", "Poignet_1", "Hanche_2", "Hanche_1",
    "Genou_2", "Genou_1", "Cheville_2", "Cheville_1"
]

# Process each image in the input folder
for image_filename in os.listdir(image_folder):
    if image_filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        # Full path to the image
        image_path = os.path.join(image_folder, image_filename)
        
        # Run pose detection (max 1 person per image)
        results = model(image_path, stream=True, max_det=1)
        
        # Extract base filename without extension
        base_filename = os.path.splitext(image_filename)[0]
        
        # Define output CSV path
        csv_filename = os.path.join(output_folder, f"{base_filename}-poses_detectees.csv")
        
        # Open CSV file for writing
        with open(csv_filename, mode='w', newline='') as csv_file:
            writer = csv.writer(csv_file)
            
            # Write CSV header
            header = ['nom_image', 'bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax']
            for kp_name in keypoints_names:
                header.append(f"{kp_name}_x")
                header.append(f"{kp_name}_y")
            writer.writerow(header)
            
            # Process detection results
            for result in results:
                for box in result.boxes:
                    # Extract bounding box coordinates (xyxy format)
                    bbox = box.xyxy[0].tolist()
                    
                    # Extract keypoints data: list of [x, y, confidence]
                    keypoints = result.keypoints.data[0].tolist()
                    
                    # Build output row
                    row = [image_filename, bbox[0], bbox[1], bbox[2], bbox[3]]
                    
                    # Initialize dictionary for keypoints
                    keypoints_dict = {kp_name: [None, None] for kp_name in keypoints_names}
                    
                    # Fill dictionary with detected keypoint coordinates
                    for idx, kp in enumerate(keypoints):
                        kp_name = keypoints_names[idx]
                        keypoints_dict[kp_name] = kp[:2]  # Keep only x and y
                    
                    # Add keypoints to row
                    for kp_name in keypoints_names:
                        row.extend(keypoints_dict[kp_name])
                    
                    # Write row to CSV
                    writer.writerow(row)

print("Done")
